# Jelastic Billing Export

This notebook connects to your **SaveInCloud / Jelastic** account, pulls billing data for all environments, exports it to CSV, and optionally uploads via SFTP.

**How to use:**
1. Fill in your credentials in Section 1
2. Run all cells (Runtime > Run all)
3. Review the results and download the CSV

---

## 0. Install Dependencies

In [ ]:
!pip install requests paramiko pyyaml -q

## 1. Configuration

Edit the values below with your credentials. Only `API_TOKEN` is required to fetch billing data. SFTP fields are optional (for uploading the CSV to your server).

In [ ]:
# =====================================================
# JELASTIC API SETTINGS
# =====================================================
# Your SaveInCloud API URL
API_URL = "https://app.paas.saveincloud.net.br/1.0/"

# Your API token (generate in SaveInCloud dashboard > Settings > API Tokens)
API_TOKEN = "YOUR_TOKEN_HERE"  # <-- PASTE YOUR TOKEN HERE

# =====================================================
# BILLING SETTINGS
# =====================================================
# How many days back to fetch (e.g. 7, 14, 30)
LOOKBACK_DAYS = 7

# Or use fixed dates (leave empty to use LOOKBACK_DAYS)
FIXED_START = ""  # e.g. "2026-05-01 00:00:00"
FIXED_END = ""    # e.g. "2026-05-12 23:59:59"

# Granularity: "HOUR", "DAY", "MONTH"
PERIOD = "DAY"

# Group nodes together? True or False
GROUP_NODES = True

# =====================================================
# SFTP UPLOAD (OPTIONAL)
# =====================================================
SFTP_ENABLED = False  # Set to True to enable upload
SFTP_HOST = "YOUR_SFTP_HOST"
SFTP_PORT = 22
SFTP_USER = "YOUR_SFTP_USER"
SFTP_PASS = "YOUR_SFTP_PASS"
SFTP_REMOTE_DIR = "/home/arthur/"

print("Configuration loaded.")

## 2. Setup (date range, API client, helpers)

Run this cell to initialize the date range and API functions.

In [ ]:
import requests
import csv
import io
import os
from datetime import datetime, timedelta

# API interval limits per period
PERIOD_MAX_DAYS = {"HOUR": 1, "DAY": 7, "MONTH": 365, "YEAR": 3650}

# Compute date range
if FIXED_START and FIXED_END:
    start_dt = datetime.strptime(FIXED_START, "%Y-%m-%d %H:%M:%S")
    end_dt = datetime.strptime(FIXED_END, "%Y-%m-%d %H:%M:%S")
else:
    now = datetime.now()
    end_dt = now.replace(hour=23, minute=59, second=59, microsecond=0)
    start_dt = (now - timedelta(days=LOOKBACK_DAYS)).replace(
        hour=0, minute=0, second=0, microsecond=0
    )

print(f"Date range: {start_dt} to {end_dt}")


def chunk_date_range(start, end, period):
    """Split date range into API-safe chunks."""
    max_days = PERIOD_MAX_DAYS.get(period, 7)
    chunks = []
    current = start
    while current < end:
        chunk_end = min(current + timedelta(days=max_days), end)
        chunks.append((current, chunk_end))
        current = chunk_end + timedelta(seconds=1)
    return chunks


def api_call(endpoint, params=None):
    """Make a Jelastic API call."""
    url = f"{API_URL.rstrip('/')}/{endpoint}"
    if params is None:
        params = {}
    params["session"] = API_TOKEN
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if data.get("result") != 0:
        print(f"  API error: {data.get('error', 'Unknown')}")
        return None
    return data


print("Setup complete.")

## 3. List Environments

Fetch all environments from your Jelastic account.

In [ ]:
data = api_call("environment/control/rest/getenvs")
environments = []

if data:
    for info in data.get("infos", []):
        env = info.get("env", {})
        nodes = info.get("nodes", [])
        env_info = {
            "envName": env.get("envName", ""),
            "appid": env.get("appid", ""),
            "domain": env.get("domain", ""),
            "status": env.get("status", 0),
            "nodes": len(nodes),
        }
        environments.append(env_info)
        status_map = {1: "Running", 2: "Stopped", 3: "Sleeping"}
        status_str = status_map.get(env_info["status"], str(env_info["status"]))
        print(f"  {env_info['envName']}")
        print(f"    Domain: {env_info['domain']}")
        print(f"    Status: {status_str}")
        print(f"    Nodes:  {env_info['nodes']}")
        print()

print(f"Total: {len(environments)} environment(s)")

## 4. Fetch Billing Data

Pull billing records for each environment. The script automatically splits large date ranges into chunks to respect API limits.

In [ ]:
all_billing_data = []

for env in environments:
    env_name = env["envName"]
    print(f"Fetching billing for: {env_name}")

    chunks = chunk_date_range(start_dt, end_dt, PERIOD)
    env_records = []

    for chunk_start, chunk_end in chunks:
        params = {
            "startTime": chunk_start.strftime("%Y-%m-%d %H:%M:%S"),
            "endTime": chunk_end.strftime("%Y-%m-%d %H:%M:%S"),
            "period": PERIOD,
            "envName": env_name,
            "groupNodes": str(GROUP_NODES).lower(),
        }
        result = api_call(
            "billing/account/rest/getaccountbillinghistorybyperiod",
            params
        )
        if result:
            items = result.get("array", [])
            for item in items:
                if not item.get("envName"):
                    item["envName"] = env_name
            env_records.extend(items)

    print(f"  Records found: {len(env_records)}")
    all_billing_data.extend(env_records)

print(f"\nTotal billing records: {len(all_billing_data)}")

if not all_billing_data:
    print("\nNo billing data found. This is normal for trial/collaborator accounts.")
    print("Use your production token to see real billing data.")

## 5. Display Billing Data

Show the billing data as a table.

In [ ]:
if all_billing_data:
    # Try to use pandas for nice display
    try:
        import pandas as pd
        df = pd.DataFrame(all_billing_data)
        # Reorder columns with most important first
        priority_cols = ["envName", "date", "resourceName", "cost", "nodeType", "nodeGroup"]
        cols = [c for c in priority_cols if c in df.columns]
        cols += [c for c in df.columns if c not in cols]
        df = df[cols]
        if "cost" in df.columns:
            df["cost"] = pd.to_numeric(df["cost"], errors="coerce")
        display(df)
        print(f"\nTotal cost: {df['cost'].sum():.6f}")
    except ImportError:
        # Fallback: plain text table
        print(f"{'envName':<15} {'date':<12} {'resourceName':<25} {'cost':<12}")
        print("-" * 70)
        total = 0
        for item in all_billing_data:
            cost = float(item.get("cost", 0))
            total += cost
            print(
                f"{item.get('envName',''):<15} "
                f"{item.get('date',''):<12} "
                f"{item.get('resourceName',''):<25} "
                f"{cost:<12.6f}"
            )
        print("-" * 70)
        print(f"Total cost: {total:.6f}")
else:
    print("No data to display.")
    print("If you expected billing data, check that:")
    print("  1. Your token has billing read permissions")
    print("  2. The account is not a trial/collaborator account")
    print("  3. The date range contains days with actual resource usage")

## 6. Export to CSV

Save the billing data to a CSV file. In Google Colab, the file will also be available for download.

In [ ]:
CSV_HEADERS = ["envName", "date", "resourceName", "cost", "nodeType", "nodeGroup", "note"]

# Build filename
csv_filename = f"billing_{datetime.now().strftime('%Y-%m-%d')}.csv"

if all_billing_data:
    # Discover all keys
    all_keys = list(CSV_HEADERS)
    for item in all_billing_data:
        for k in item:
            if k not in all_keys:
                all_keys.append(k)

    with open(csv_filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=all_keys, extrasaction="ignore")
        writer.writeheader()
        for item in all_billing_data:
            row = {}
            for key in all_keys:
                val = item.get(key, "")
                if key == "cost" and val != "":
                    try:
                        val = f"{float(val):.6f}"
                    except (ValueError, TypeError):
                        pass
                row[key] = val
            writer.writerow(row)

    print(f"CSV saved: {csv_filename} ({len(all_billing_data)} rows)")

    # Auto-download in Google Colab
    try:
        from google.colab import files
        files.download(csv_filename)
        print("Download started automatically.")
    except ImportError:
        print(f"Not in Colab. File saved locally: {csv_filename}")
else:
    print("No billing data to export.")

## 7. Upload via SFTP (Optional)

Upload the CSV to your server. Only runs if `SFTP_ENABLED = True` in Section 1.

In [ ]:
if SFTP_ENABLED and all_billing_data:
    import paramiko

    print(f"Connecting to SFTP: {SFTP_USER}@{SFTP_HOST}:{SFTP_PORT}")

    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

    try:
        ssh.connect(
            hostname=SFTP_HOST,
            port=SFTP_PORT,
            username=SFTP_USER,
            password=SFTP_PASS,
            timeout=30,
        )
        sftp = ssh.open_sftp()

        # Ensure remote directory exists
        for d in SFTP_REMOTE_DIR.strip("/").split("/"):
            if not d:
                continue
            try:
                sftp.stat(f"/{d}")
            except FileNotFoundError:
                sftp.mkdir(f"/{d}")

        remote_path = f"{SFTP_REMOTE_DIR.rstrip('/')}/{csv_filename}"
        sftp.put(csv_filename, remote_path)
        print(f"Uploaded: {csv_filename} -> {remote_path}")

        # Verify
        attrs = sftp.stat(remote_path)
        print(f"Verified: {attrs.st_size} bytes on server")

        sftp.close()
        ssh.close()
        print("SFTP upload complete!")

    except Exception as e:
        print(f"SFTP upload failed: {e}")
        try:
            ssh.close()
        except:
            pass

elif not SFTP_ENABLED:
    print("SFTP upload disabled. Set SFTP_ENABLED = True in Section 1 to enable.")
else:
    print("No data to upload.")

## 8. Cost Summary by Environment

Show a breakdown of costs per environment and resource type.

In [ ]:
if all_billing_data:
    # Summary by environment
    env_totals = {}
    resource_totals = {}

    for item in all_billing_data:
        env_name = item.get("envName", "unknown")
        resource = item.get("resourceName", "unknown")
        cost = float(item.get("cost", 0))

        env_totals[env_name] = env_totals.get(env_name, 0) + cost
        key = (env_name, resource)
        resource_totals[key] = resource_totals.get(key, 0) + cost

    print("COST BY ENVIRONMENT")
    print("=" * 50)
    grand_total = 0
    for env_name, total in sorted(env_totals.items()):
        print(f"  {env_name:<30} {total:.6f}")
        grand_total += total
    print(f"  {'TOTAL':<30} {grand_total:.6f}")

    print()
    print("COST BY RESOURCE TYPE")
    print("=" * 60)
    current_env = None
    for (env_name, resource), total in sorted(resource_totals.items()):
        if env_name != current_env:
            if current_env is not None:
                print()
            print(f"  [{env_name}]")
            current_env = env_name
        print(f"    {resource:<35} {total:.6f}")
else:
    print("No billing data to summarize.")

---

## Next Steps

Once you're satisfied with the output:

1. **For automated weekly runs**, install the standalone script on your Linux VM:
   - Clone: `git clone https://github.com/anirudhatalmale6-alt/jelastic-billing-export.git`
   - Edit `config.yaml` with your production credentials
   - Schedule with cron: `0 6 * * 1 cd /path/to/script && python3 jelastic_billing_export.py`

2. **To change the date range**, edit `LOOKBACK_DAYS` or set `FIXED_START`/`FIXED_END` in Section 1.

3. **To get hourly data**, set `PERIOD = "HOUR"` (max 1 day range per chunk).

4. **To export per environment**, run the standalone script with `csv.per_environment: true` in config.yaml.